# PyTorch Geometric: Real-World GNN Applications

## What is PyTorch Geometric?

PyTorch Geometric (PyG) is the **industry-standard library** for Graph Neural Networks.

### Key Features
✅ Pre-implemented GNN layers (GCN, GAT, GraphSAGE, etc.)  
✅ Built-in benchmark datasets (Cora, Citeseer, OGB, etc.)  
✅ Efficient sparse tensor operations  
✅ Mini-batch sampling and DataLoader  
✅ GPU acceleration ready  
✅ Active community and excellent documentation  

### When to Use PyG vs. Custom Implementation
- **Custom**: Learning, research, novel architectures
- **PyG**: Production, benchmarking, prototyping
- **Both**: Learn with custom, then switch to PyG for efficiency

## Installation & Setup

```bash
# Install PyTorch first
pip install torch

# Install PyTorch Geometric
pip install torch-geometric

# Optional: GPU support
pip install torch-geometric[torch-cuda11.8]
```

## Overview of This Module

1. **Dataset Loading & Exploration**: Working with real citation networks
2. **Benchmarking GNNs**: Comparing architectures on standard datasets
3. **Mini-batch Training**: Scalable training strategies
4. **Transfer Learning**: Pre-trained models and fine-tuning
5. **Model Evaluation**: Comprehensive metrics and analysis
6. **Deployment**: Saving and loading models

---

# Part 1: Dataset Loading and Exploration

In [ ]:
"""
PyTorch Geometric: Real-World GNN Applications
Module 3: Using PyG for Production GNNs
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid, OGB
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, global_mean_pool
from torch_geometric.utils import degree
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("PYTORCH GEOMETRIC: REAL-WORLD GNN APPLICATIONS")
print("=" * 70)
print(f"\n✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Device: {device}")

# ============================================================================
# Part 1: Load Citation Network (Cora)
# ============================================================================

print("\n" + "=" * 70)
print("PART 1: CITATION NETWORKS")
print("=" * 70)

# Download and load Cora dataset
print("\n📥 Loading Cora dataset...")
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]

print(f"""
📊 CORA DATASET:
   • Nodes: {data.num_nodes:,}
   • Edges: {data.num_edges:,}
   • Features per node: {data.num_features}
   • Classes: {dataset.num_classes}
   • Density: {data.num_edges / (data.num_nodes * (data.num_nodes - 1)):.4f}
   
📈 Edge Statistics:
   • Average degree: {(2 * data.num_edges / data.num_nodes):.2f}
   • Min degree: {degree(data.edge_index[0]).min().item():.0f}
   • Max degree: {degree(data.edge_index[0]).max().item():.0f}
   
✅ Dataset loaded successfully!
""")

# Visualize degree distribution
degrees = degree(data.edge_index[0], num_nodes=data.num_nodes)
print(f"📊 Degree Distribution Statistics:")
print(f"   • Mean: {degrees.float().mean():.2f}")
print(f"   • Median: {degrees.float().median():.2f}")
print(f"   • Std: {degrees.float().std():.2f}")

# Plot degree distribution
plt.figure(figsize=(10, 4))
plt.hist(degrees.numpy(), bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Node Degree', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Degree Distribution in Cora Network', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("""
🔍 Interpretation:
   • Most nodes have degree 1-10
   • Few hub nodes with very high degree (power-law distribution)
   • Typical of citation networks
""")

In [ ]:
# ============================================================================
# Part 2: Build GNN Models Using PyG Layers
# ============================================================================

print("\n" + "=" * 70)
print("PART 2: GNN MODELS WITH PYTORCH GEOMETRIC")
print("=" * 70)

class GCNModelPyG(torch.nn.Module):
    """
    Graph Convolutional Network using PyTorch Geometric layers.
    
    Advantages over custom implementation:
    - Optimized C++ backend
    - Automatic GPU acceleration
    - Handles sparse tensors efficiently
    - Well-tested and debugged
    """
    
    def __init__(self, in_channels: int, hidden_channels: int, 
                 num_classes: int, num_layers: int = 2, dropout: float = 0.5):
        super().__init__()
        self.in_channels = in_channels
        self.hidden_channels = hidden_channels
        self.num_classes = num_classes
        self.num_layers = num_layers
        self.dropout = dropout
        
        # Layer list
        self.convs = torch.nn.ModuleList()
        
        # First layer
        self.convs.append(GCNConv(in_channels, hidden_channels))
        
        # Hidden layers
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
        
        # Output layer
        self.convs.append(GCNConv(hidden_channels, num_classes))
    
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.
        
        Args:
            x: (num_nodes, num_features) node features
            edge_index: (2, num_edges) edge indices
            
        Returns:
            logits: (num_nodes, num_classes)
        """
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        
        # Output layer (no activation)
        x = self.convs[-1](x, edge_index)
        return x


class GATModelPyG(torch.nn.Module):
    """
    Graph Attention Network using PyTorch Geometric.
    
    Multi-head attention with PyG's optimized implementation.
    """
    
    def __init__(self, in_channels: int, hidden_channels: int,
                 num_classes: int, num_heads: int = 8, num_layers: int = 2,
                 dropout: float = 0.2):
        super().__init__()
        self.in_channels = in_channels
        self.hidden_channels = hidden_channels
        self.num_classes = num_classes
        self.num_heads = num_heads
        self.dropout = dropout
        
        self.convs = torch.nn.ModuleList()
        
        # First layer
        self.convs.append(GATConv(in_channels, hidden_channels, heads=num_heads,
                                  dropout=dropout))
        
        # Hidden layers
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(num_heads * hidden_channels, hidden_channels,
                                      heads=num_heads, dropout=dropout))
        
        # Output layer (single head)
        self.convs.append(GATConv(num_heads * hidden_channels, num_classes,
                                  heads=1, dropout=dropout))
    
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, edge_index)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        
        x = self.convs[-1](x, edge_index)
        return x


class GraphSAGEModelPyG(torch.nn.Module):
    """
    GraphSAGE using PyTorch Geometric.
    
    Supports inductive learning and sampling.
    """
    
    def __init__(self, in_channels: int, hidden_channels: int,
                 num_classes: int, num_layers: int = 2, dropout: float = 0.5):
        super().__init__()
        self.in_channels = in_channels
        self.hidden_channels = hidden_channels
        self.dropout = dropout
        
        self.convs = torch.nn.ModuleList()
        
        self.convs.append(SAGEConv(in_channels, hidden_channels))
        for _ in range(num_layers - 2):
            self.convs.append(SAGEConv(hidden_channels, hidden_channels))
        self.convs.append(SAGEConv(hidden_channels, num_classes))
    
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        
        x = self.convs[-1](x, edge_index)
        return x


# Create models
print("\n📦 Creating models...")
gcn_model = GCNModelPyG(dataset.num_features, 64, dataset.num_classes,
                        num_layers=2, dropout=0.5).to(device)
gat_model = GATModelPyG(dataset.num_features, 64, dataset.num_classes,
                        num_heads=8, num_layers=2, dropout=0.2).to(device)
sage_model = GraphSAGEModelPyG(dataset.num_features, 64, dataset.num_classes,
                               num_layers=2, dropout=0.5).to(device)

print(f"\n✓ Models created successfully!")
print(f"   • GCN parameters: {sum(p.numel() for p in gcn_model.parameters()):,}")
print(f"   • GAT parameters: {sum(p.numel() for p in gat_model.parameters()):,}")
print(f"   • GraphSAGE parameters: {sum(p.numel() for p in sage_model.parameters()):,}")

# Test forward pass
data = data.to(device)
with torch.no_grad():
    gcn_out = gcn_model(data.x, data.edge_index)
    gat_out = gat_model(data.x, data.edge_index)
    sage_out = sage_model(data.x, data.edge_index)

print(f"\n✓ Forward pass successful!")
print(f"   • Output shape: {gcn_out.shape}")

In [ ]:
# ============================================================================
# Part 3: Training on Real Data
# ============================================================================

print("\n" + "=" * 70)
print("PART 3: TRAINING GNN MODELS")
print("=" * 70)

def train_model(model, data, epochs=100, lr=0.01, patience=20):
    """
    Train a GNN model on Cora dataset.
    
    Args:
        model: GNN model
        data: Dataset
        epochs: Number of training epochs
        lr: Learning rate
        patience: Early stopping patience
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.CrossEntropyLoss()
    
    best_val_acc = 0
    patience_counter = 0
    history = {'train_loss': [], 'val_acc': [], 'test_acc': []}
    
    print(f"\n🚀 Training {model.__class__.__name__}...")
    print(f"{'Epoch':<6} {'Train Loss':<12} {'Val Acc':<12} {'Test Acc':<12} {'Status':<10}")
    print("-" * 60)
    
    for epoch in range(epochs):
        # Training
        model.train()
        optimizer.zero_grad()
        
        out = model(data.x, data.edge_index)
        loss = criterion(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
        
        # Validation
        model.eval()
        with torch.no_grad():
            out = model(data.x, data.edge_index)
            val_acc = (out[data.val_mask].argmax(1) == data.y[data.val_mask]).float().mean()
            test_acc = (out[data.test_mask].argmax(1) == data.y[data.test_mask]).float().mean()
        
        history['train_loss'].append(loss.item())
        history['val_acc'].append(val_acc.item())
        history['test_acc'].append(test_acc.item())
        
        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc.item()
            patience_counter = 0
            status = "✓"
        else:
            patience_counter += 1
            status = ""
        
        if (epoch + 1) % 20 == 0:
            print(f"{epoch+1:<6} {loss.item():<12.4f} {val_acc:<12.4f} "
                  f"{test_acc:<12.4f} {status:<10}")
        
        if patience_counter >= patience:
            break
    
    return history, best_val_acc

# Train GCN model
print("\n" + "=" * 70)
print("Training GCN on Cora")
print("=" * 70)
gcn_history, gcn_best_acc = train_model(gcn_model, data, epochs=100, lr=0.01, patience=20)

# Train GAT model
print("\n" + "=" * 70)
print("Training GAT on Cora")
print("=" * 70)
gat_history, gat_best_acc = train_model(gat_model, data, epochs=100, lr=0.01, patience=20)

# Train GraphSAGE model
print("\n" + "=" * 70)
print("Training GraphSAGE on Cora")
print("=" * 70)
sage_history, sage_best_acc = train_model(sage_model, data, epochs=100, lr=0.01, patience=20)

# ============================================================================
# Part 4: Evaluation and Comparison
# ============================================================================

print("\n" + "=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)

results = {
    'GCN': (gcn_history, gcn_best_acc),
    'GAT': (gat_history, gat_best_acc),
    'GraphSAGE': (sage_history, sage_best_acc)
}

print("\n📊 Model Performance Comparison:")
print(f"{'Model':<15} {'Best Val Acc':<15} {'Final Test Acc':<15}")
print("-" * 45)

for model_name, (hist, best_acc) in results.items():
    final_test_acc = hist['test_acc'][-1]
    print(f"{model_name:<15} {best_acc:<15.4f} {final_test_acc:<15.4f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Validation accuracy
ax1 = axes[0]
for model_name, (hist, _) in results.items():
    ax1.plot(hist['val_acc'], label=model_name, linewidth=2, marker='o', markersize=3)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Validation Accuracy', fontsize=12)
ax1.set_title('Validation Accuracy Over Time', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Plot 2: Training loss
ax2 = axes[1]
for model_name, (hist, _) in results.items():
    ax2.plot(hist['train_loss'], label=model_name, linewidth=2, marker='s', markersize=3)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Training Loss', fontsize=12)
ax2.set_title('Training Loss Over Time', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Training and evaluation complete!")